# 04. 데이터 전처리

> 결측치 → 이상치 → 파생변수/구간화 → 인코딩 → 스케일링 → 분할로 이어지는 정해진 순서의 파이프라인이라 하나의 파일로 유지합니다. 실제 시험에서도 전처리에 90분 중 절반 이상의 시간이 쓰인다는 점을 기억하세요.

### 이 노트북에서 배우는 것
- 결측치를 상황에 맞게(삭제/대체/보간) 처리하는 기준
- 이상치를 IQR과 Z-score 두 가지 방법으로 탐지·처리하는 방법
- 문자열에서 새로운 컬럼(파생변수)을 만들고 연속형을 구간화(binning)하는 기법
- 범주형 변수를 숫자로 바꾸는 여러 인코딩 기법의 차이
- 스케일링이 필요한 이유와 train_test_split의 원칙(stratify 등)

### 사용 데이터
- `data/train.csv` (Titanic)

## 목차
1. 결측치 탐색 및 처리
2. 이상치 탐색 및 처리
3. 파생변수 생성 & 구간화
4. 범주형 변수 인코딩
5. 피처 스케일링
6. 데이터 분할
7. 종합 실전 문제

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('../data/train.csv')
titanic_backup = df.copy()  # 원본은 항상 백업해두고 실습
df.head(3)

## 1. 결측치 탐색 및 처리

### 📖 이론 설명
결측치를 처리하지 않으면 모델 학습 시 에러가 나거나, 분석 결과가 왜곡됩니다. 처리 방법은 크게 **① 삭제(dropna)**와 **② 대체(fillna)**로 나뉩니다. 어떤 방법을 쓸지는 '결측치가 얼마나 많은가', '그 컬럼이 얼마나 중요한가'에 따라 판단합니다 - 결측치가 90% 넘는 컬럼은 대체보다 컬럼째로 삭제하는 것이 낫고, 소수의 결측치는 평균/중앙값/최빈값으로 채우는 것이 일반적입니다. 시계열/순서가 있는 데이터라면 `interpolate()`(선형 보간)나 `bfill`/`ffill`(앞뒤 값 복사)이 더 자연스럽습니다. 주의할 점은, 결측치가 **꼭 NaN으로만 표시되지 않는다**는 것입니다. `' '`(공백)처럼 '숨은 결측치'가 있으면 `astype()` 형변환이 실패하므로, `df.replace(' ', np.nan)`으로 먼저 정리해야 합니다. `sklearn.impute.SimpleImputer`로도 동일한 대체 작업을 할 수 있습니다.

### 🧩 핵심 개념 정리
| 함수 | 설명 |
|---|---|
| df.isnull().sum() | 컬럼별 결측치 개수 |
| df.dropna(axis=, how=, subset=) | 결측치 삭제 |
| df.fillna(값) | 결측치 대체 |
| df.replace(찾을값, 바꿀값) | 특정 값 치환(숨은 결측치 정리) |
| df['col'].interpolate() | 선형 보간 |
| df['col'].bfill()/.ffill() | 뒤/앞 값으로 채우기 |
| sklearn.impute.SimpleImputer | 대체를 객체로 관리(모델 파이프라인에 유리) |

### 💻 예제 코드

In [ ]:
print(df.isnull().sum())

# Age: 평균으로 대체
df['Age'] = df['Age'].fillna(df['Age'].mean())

# Embarked: 결측치가 적으므로 최빈값으로 대체
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

# Cabin: 결측치가 압도적으로 많으므로(약 77%) 컬럼째 삭제
df = df.drop(columns=['Cabin'])
print(df.isnull().sum())

### ✍️ TODO: 실전 문제

**문제 1.** `titanic_backup`에서 결측치가 있는 행을 모두 삭제한 결과의 행 개수를 확인하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
print(titanic_backup.dropna().shape[0])  # dropna()는 기본적으로 '하나라도 결측치가 있는 행'을 통째로 삭제함
```

</details>

**문제 2.** `titanic_backup`의 `Age` 결측치를 **중앙값**으로 채운 결과를 `age_median_filled`에 저장하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
age_median_filled = titanic_backup['Age'].fillna(titanic_backup['Age'].median())  # 평균보다 이상치 영향을 덜 받는 중앙값으로 대체
```

</details>

**문제 3.** `titanic_backup['TotalCharges'] = ['100', ' ', '200']`처럼 공백 문자열이 섞인 시리즈가 있다고 가정합니다. 이 값을 `np.nan`으로 바꾼 뒤 `astype(float)`로 변환해보세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
s = pd.Series(['100', ' ', '200'])
s = s.replace(' ', np.nan).astype(float)  # 공백 문자열은 isnull()로 안 잡히므로 먼저 NaN으로 바꿔야 float 변환이 가능해짐
print(s)
```

</details>

## 2. 이상치 탐색 및 처리

### 📖 이론 설명
이상치(outlier)는 나머지 데이터와 동떨어진 값으로, 모델 학습을 왜곡시킬 수 있습니다. 가장 널리 쓰이는 방법은 **IQR(사분위 범위)** 방식입니다: Q1(25%), Q3(75%)를 구하고 `IQR = Q3 - Q1`, 그 1.5배를 벗어나면 이상치로 봅니다. 이상치를 **제거**할 수도 있고, 상한/하한 값으로 눌러 대체(**clip**)할 수도 있습니다 - 데이터가 적으면 제거보다 clip이 정보 손실이 적습니다. 또 다른 방법은 **Z-score(표준화 점수)** 방식으로, 평균에서 표준편차의 몇 배 떨어져 있는지를 봅니다(보통 `|z| > 1.96`이면 상하위 약 2.5%에 해당하는 극단값으로 판단).

### 🧩 핵심 개념 정리
| 방법 | 기준 |
|---|---|
| IQR | Q1-1.5×IQR ~ Q3+1.5×IQR 벗어나면 이상치 |
| Z-score | \|((x-평균)/표준편차)\| > 1.96 이면 이상치 |

### 💻 예제 코드

In [ ]:
# IQR로 이상치를 상/하한값으로 clip
def clip_iqr_outliers(dataframe, column):
    q1, q3 = dataframe[column].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    dataframe[column] = dataframe[column].clip(lower, upper)
    return dataframe

df = clip_iqr_outliers(df, 'Fare')

# Z-score로 이상치 '개수' 확인
z_outliers = df[(abs((df['Fare'] - df['Fare'].mean()) / df['Fare'].std())) > 1.96]
print('Z-score 기준 이상치 개수:', len(z_outliers))

### ✍️ TODO: 실전 문제

**문제 1.** `titanic_backup['Age']`에서 IQR 기준 이상치의 **개수**를 구하는 함수를 만들어 실행하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
def count_iqr_outliers(s):
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr  # 1.5 * IQR은 이상치 판단에 관례적으로 쓰이는 배수
    return ((s < lower) | (s > upper)).sum()

print(count_iqr_outliers(titanic_backup['Age'].dropna()))  # quantile 계산 전 결측치(NaN)를 먼저 제거해야 함
```

</details>

**문제 2.** `titanic_backup['Fare']`에서 Z-score 기준(`|z|>1.96`) 이상치 인덱스를 찾아 제거한 결과의 행 개수를 확인하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
fare = titanic_backup['Fare']
outlier_idx = fare[(abs((fare - fare.mean()) / fare.std())) > 1.96].index  # (값-평균)/표준편차 = Z-score, |z|>1.96은 정규분포 기준 상하위 약 2.5%씩(총 5%)에 해당
removed = titanic_backup.drop(outlier_idx)  # 인덱스를 구한 뒤 drop으로 해당 행들을 원본에서 제거
print(removed.shape[0])
```

</details>

## 3. 파생변수 생성 & 구간화

### 📖 이론 설명
기존 컬럼을 그대로 쓰는 것보다, 의미 있는 정보를 뽑아 **새 컬럼(파생변수)**으로 만들면 모델 성능이 좋아지는 경우가 많습니다. 문자열 컬럼이라면 `split()`/`strip()`을 `apply(lambda ...)`와 함께 써서 원하는 조각만 뽑아냅니다. 연속형 변수는 `pd.cut()`으로 구간을 나눠 범주형처럼 다룰 수 있습니다(예: 나이 → 어린이/청소년/어른/노인).

### 🧩 핵심 개념 정리
| 함수 | 설명 |
|---|---|
| Series.apply(lambda x: ...) | 각 원소에 커스텀 함수 적용 |
| str.split(sep)[i] / str.strip() | 문자열 자르기 / 공백 제거 |
| pd.cut(컬럼, bins=, labels=) | 연속형 → 구간형(범주) 변환 |

### 💻 예제 코드

In [ ]:
# 이름에서 호칭(Title) 추출: 'Braund, Mr. Owen Harris' -> 'Mr'
df['Title'] = df['Name'].apply(lambda x: x.split(',')[1].split('.')[0].strip())
print(df['Title'].value_counts().head())

# 나이를 구간으로 나누기
bins = [0, 12, 19, 59, np.inf]
labels = ['어린이', '청소년', '어른', '노인']
df['AgeGroup'] = pd.cut(df['Age'], bins=bins, labels=labels)
print(df[['Age', 'AgeGroup']].head())

### ✍️ TODO: 실전 문제

**문제 1.** `df['Title']`에서 'Mr', 'Mrs', 'Miss', 'Master'가 아닌 값은 모두 'Rare'로 바꿔 `Title` 컬럼을 업데이트하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
common = ['Mr', 'Mrs', 'Miss', 'Master']
df['Title'] = df['Title'].apply(lambda t: t if t in common else 'Rare')  # 목록에 없는 소수 값들을 'Rare'로 묶어 카테고리 수를 줄임(인코딩 시 컬럼 폭발 방지)
df['Title'].value_counts()
```

</details>

**문제 2.** `Fare`를 [0, 10, 50, 100, np.inf] 기준으로 'low','mid','high','very_high' 구간으로 나눠 `FareGroup` 컬럼을 만드세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
df['FareGroup'] = pd.cut(df['Fare'], bins=[0, 10, 50, 100, np.inf], labels=['low', 'mid', 'high', 'very_high'])  # np.inf로 마지막 구간을 열어두면 최댓값을 몰라도 상한 없이 처리 가능
df['FareGroup'].value_counts()
```

</details>

## 4. 범주형 변수 인코딩

### 📖 이론 설명
머신러닝 모델은 문자열을 이해하지 못하므로 범주형 데이터를 숫자로 바꿔야 합니다. 방법은 크게 두 갈래입니다.

**① 레이블 인코딩(순서를 부여)**: `LabelEncoder`, `factorize()`, `astype('category').cat.codes`는 모두 각 범주에 정수를 하나씩 매핑합니다. 다만 이 방식은 모델이 '숫자가 크면 더 큰 값'이라고 오해할 수 있어, 순서가 없는 범주(예: 성별, 지역)에는 주의가 필요합니다. `LabelEncoder`는 `.inverse_transform()`으로 다시 원래 문자열로 되돌릴 수 있습니다.

**② 원-핫 인코딩(순서 없음)**: `pd.get_dummies()` 또는 `OneHotEncoder`는 각 범주를 0/1로 된 별도 컬럼으로 펼쳐서, 모델이 범주 간 크기 비교를 하지 않도록 만듭니다. `drop_first=True`는 다중공선성을 줄이기 위해 첫 번째 범주 컬럼을 생략하는 옵션입니다. `OneHotEncoder`는 결과를 기본적으로 **희소행렬(sparse matrix)**로 반환하므로, DataFrame으로 보려면 `.toarray()`로 밀집행렬로 바꿔야 하고, 각 컬럼이 어떤 범주였는지는 `.categories_`로 확인합니다.

### 🧩 핵심 개념 정리
| 함수 | 방식 | 특징 |
|---|---|---|
| LabelEncoder | 레이블 | .inverse_transform()으로 디코딩 가능 |
| pd.factorize() | 레이블 | 등장 순서대로 정수 부여, uniques 함께 반환 |
| astype('category').cat.codes | 레이블 | 판다스 카테고리 타입 활용 |
| pd.get_dummies(drop_first=) | 원-핫 | 컬럼이 여러 개로 늘어남 |
| OneHotEncoder | 원-핫 | sparse 결과, .toarray(), .categories_ |

### 💻 예제 코드

In [ ]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

# 레이블 인코딩
le = LabelEncoder()
df['Sex_label'] = le.fit_transform(df['Sex'])
print(df[['Sex', 'Sex_label']].drop_duplicates())
print(le.inverse_transform(df['Sex_label'])[:3])  # 다시 문자열로 디코딩

# factorize
labels, uniques = pd.factorize(df['Embarked'])
print(uniques)

# 원-핫 인코딩 (pandas)
df_dummy = pd.get_dummies(df, columns=['Embarked'], drop_first=True)
print([c for c in df_dummy.columns if 'Embarked' in c])

# 원-핫 인코딩 (sklearn) - 희소행렬 -> 밀집행렬
ohe = OneHotEncoder()
encoded = ohe.fit_transform(df[['Embarked']]).toarray()
print(ohe.categories_)

### ✍️ TODO: 실전 문제

**문제 1.** `df['Sex']`를 `astype('category').cat.codes`로 숫자화한 결과를 `sex_catcode`에 저장하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
sex_catcode = df['Sex'].astype('category').cat.codes  # category 타입으로 바꾼 뒤 cat.codes를 쓰면 LabelEncoder 없이도 문자열을 정수로 변환 가능
print(sex_catcode.head())
```

</details>

**문제 2.** `df['Pclass']`를 `pd.get_dummies(drop_first=True)`로 원-핫 인코딩하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
pd.get_dummies(df, columns=['Pclass'], drop_first=True).filter(like='Pclass')  # drop_first=True: 다중공선성을 피하기 위해 카테고리 중 하나를 기준(base)으로 삼고 나머지만 컬럼으로 만듦
```

</details>

**문제 3.** `OneHotEncoder`로 `df[['Sex']]`를 인코딩한 뒤, 어떤 컬럼이 어느 범주였는지 `categories_`로 확인하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
ohe2 = OneHotEncoder()
encoded2 = ohe2.fit_transform(df[['Sex']]).toarray()  # fit_transform 결과가 희소행렬(sparse)로 나오므로 toarray()로 일반 배열로 변환
print(ohe2.categories_)  # 몇 번째 열이 어떤 범주였는지는 categories_로 확인해야 함(순서가 자동 지정되기 때문)
print(encoded2[:3])
```

</details>

## 5. 피처 스케일링

### 📖 이론 설명
컬럼마다 값의 범위(스케일)가 크게 다르면(예: `Age`는 0~80, `Fare`는 0~500), 거리 기반 모델(KNN, SVM)이나 경사하강법 기반 모델(딥러닝, 로지스틱회귀)은 값이 큰 컬럼에 더 큰 영향을 받아 학습이 왜곡될 수 있습니다. 스케일러는 이 범위를 맞춰줍니다.

- **StandardScaler**: 평균 0, 표준편차 1로 표준화. 데이터가 정규분포에 가까울 때 적합.
- **MinMaxScaler**: 최소값 0, 최대값 1 사이로 변환. 값의 범위를 명확히 제한하고 싶을 때.
- **RobustScaler**: 평균/표준편차 대신 중앙값/IQR을 사용해 **이상치의 영향을 덜 받음**.

**중요한 원칙**: `fit_transform()`은 학습 데이터(train)에만 적용하고, 테스트 데이터(test)에는 학습 데이터에서 이미 계산된 값으로 `transform()`만 해야 합니다. 그래야 테스트 데이터의 정보가 학습 과정에 새어 들어가는 '데이터 누수(leakage)'를 막을 수 있습니다.

### 🧩 핵심 개념 정리
| 스케일러 | 특징 |
|---|---|
| StandardScaler | 평균0, 표준편차1 |
| MinMaxScaler | 0~1 범위로 압축 |
| RobustScaler | 이상치에 강함(중앙값·IQR 사용) |

### 💻 예제 코드

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

num_cols = ['Age', 'Fare']
scaler = StandardScaler()
df_scaled = df.copy()
df_scaled[num_cols] = scaler.fit_transform(df[num_cols])
print(df_scaled[num_cols].describe())

### ✍️ TODO: 실전 문제

**문제 1.** `Fare` 컬럼을 `MinMaxScaler`로 변환한 결과의 최소/최대값을 확인하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
mm = MinMaxScaler()
fare_scaled = mm.fit_transform(df[['Fare']])  # fit_transform 입력은 2차원이어야 하므로 df[['Fare']]처럼 대괄호를 두 번 써서 DataFrame 형태로 전달
print(fare_scaled.min(), fare_scaled.max())  # MinMaxScaler는 값을 정확히 0~1 사이로 압축하는 것이 목적
```

</details>

**문제 2.** `Fare` 컬럼(이상치가 많은 컬럼)을 `RobustScaler`로 변환해보고 `StandardScaler` 결과와 분포 차이를 비교하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
rs = RobustScaler()
fare_robust = rs.fit_transform(df[['Fare']])  # RobustScaler는 평균/표준편차 대신 중앙값과 IQR을 사용해 이상치의 영향을 덜 받음
print(pd.Series(fare_robust.flatten()).describe())
```

</details>

## 6. 데이터 분할

### 📖 이론 설명
모델을 학습시킬 데이터(train)와 성능을 평가할 데이터(test)는 반드시 나눠야 합니다 - 학습에 쓴 데이터로 평가하면 '답을 보고 채점하는' 것과 같아서 성능이 과장됩니다. `stratify=y`를 주면 분할 후에도 클래스 비율(예: 생존/사망 비율)이 원본과 비슷하게 유지되어, 불균형 데이터에서 특히 중요합니다. 분할 후 인덱스가 뒤섞이므로 `reset_index(drop=True)`로 정리해두면 이후 작업이 편해집니다.

### 🧩 핵심 개념 정리
| 옵션 | 설명 |
|---|---|
| test_size | 테스트 데이터 비율 |
| stratify=y | y의 클래스 비율을 유지해서 분할 |
| random_state | 매번 같은 분할 결과가 나오도록 고정 |

### 💻 예제 코드

In [ ]:
from sklearn.model_selection import train_test_split

X = df_scaled[['Pclass', 'Age', 'Fare']]
y = df_scaled['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
print(X_train.shape, X_test.shape)
print(y_train.mean(), y_test.mean())  # stratify 덕분에 비율이 비슷함

### ✍️ TODO: 실전 문제

**문제 1.** `X`, `y`를 `test_size=0.3`, `stratify=y`, `random_state=123`으로 분할하고 각 세트의 shape을 출력하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
X_train2, X_test2, y_train2, y_test2 = train_test_split(X, y, test_size=0.3, stratify=y, random_state=123)  # stratify=y: 분할 후에도 train/test의 클래스 비율이 원본과 비슷하게 유지되도록 강제
print(X_train2.shape, X_test2.shape)
```

</details>

## 7. 종합 실전 문제

**전처리 전체 파이프라인을 순서대로 이어서 풀어보는 미니 모의고사입니다.**

**문제 1.** `titanic_backup`을 복사해 `t`에 저장하고, `Age`는 평균으로, `Embarked`는 최빈값으로 결측치를 채우고, `Cabin`은 삭제하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
t = titanic_backup.copy()  # copy()로 복사해야 원본 titanic_backup이 바뀌지 않음
t['Age'] = t['Age'].fillna(t['Age'].mean())
t['Embarked'] = t['Embarked'].fillna(t['Embarked'].mode()[0])  # 범주형은 평균을 낼 수 없으므로 최빈값(mode)으로 채움. mode()는 Series를 반환하므로 [0]으로 첫 값을 꺼냄
t = t.drop(columns=['Cabin'])  # 결측치 비율이 너무 높은 컬럼은 채우기보다 통째로 제거하는 것이 나을 때가 많음
print(t.isnull().sum())
```

</details>

**문제 2.** `t`의 `Sex`, `Embarked`를 `get_dummies(drop_first=True)`로 원-핫 인코딩하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
t = pd.get_dummies(t, columns=['Sex', 'Embarked'], drop_first=True)  # 여러 컬럼을 리스트로 한 번에 지정해 동시에 원-핫 인코딩
t.head()
```

</details>

**문제 3.** `t`에서 `Survived`를 y로, 나머지 수치형 컬럼(`Name`,`Ticket` 등 텍스트 컬럼 제외)을 X로 만들어 `train_test_split(stratify=y)`으로 나누세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
feature_cols = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare'] + [c for c in t.columns if c.startswith('Sex_') or c.startswith('Embarked_')]  # get_dummies로 생긴 컬럼명은 'Sex_male'처럼 접두어가 붙으므로 startswith로 찾아 리스트에 합침
X = t[feature_cols]
y = t['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)  # stratify=y로 Survived 비율을 train/test에 동일하게 유지
print(X_train.shape, X_test.shape)
```

</details>